# Position-Bias / Middle-Context Retrieval — Colab T4 runner

**Before running:** Runtime → Change runtime type → **T4 GPU**.

This notebook runs the full Phase-1 pipeline and shows the plots. Edit the
`EXAMPLES_PER_POSITION` / `CONTEXT_LENGTH` constants below to scale up.

In [ ]:
# 0. Confirm GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 1. Get the code onto Colab.
#    Option A: clone your repo  ->  !git clone <URL> repo && %cd repo
#    Option B: upload a zip of this project, then unzip:
# from google.colab import files; up = files.upload()
# !unzip -o *.zip -d . && %cd <unzipped-folder>
import os; print('cwd:', os.getcwd()); print(os.listdir('.'))

In [ ]:
# 2. Dependencies (Colab already has torch + CUDA)
!pip install -q transformers scipy matplotlib pyyaml

In [ ]:
# 3. Scale knobs
EXAMPLES_PER_POSITION = 50   # 50 first real run; 100 for better statistics
CONTEXT_LENGTH = 1024        # 1024 first; 2048 if time permits
CFG = 'configs/default.yaml'

In [ ]:
# 4. Generate dataset
!python scripts/01_generate_dataset.py --config $CFG --examples-per-position $EXAMPLES_PER_POSITION --context-length $CONTEXT_LENGTH

In [ ]:
# IMPORTANT: scripts 02-06 locate the dataset via context_length + examples_per_position
# in the config. So bake the same values into the config for the rest of the run.
import yaml
with open(CFG) as f: cfg = yaml.safe_load(f)
cfg['data']['examples_per_position'] = EXAMPLES_PER_POSITION
cfg['data']['context_length'] = CONTEXT_LENGTH
with open(CFG, 'w') as f: yaml.safe_dump(cfg, f)
print('config updated')

In [ ]:
# 5. Experiment 1 + 2 (baseline accuracy & influence)
!python scripts/02_eval_baseline.py     --config $CFG
!python scripts/03_measure_influence.py --config $CFG

In [ ]:
# 6. Experiment 3 + 4 (standard fine-tune & middle-weighted intervention)
!python scripts/04_train_standard.py        --config $CFG
!python scripts/05_train_middle_weighted.py --config $CFG

In [ ]:
# 7. Report + comparison plots
!python scripts/06_make_report.py --config $CFG
print(open('outputs/results_summary.md').read())

In [ ]:
# 7b. Step-0 experiment (beats arXiv:2603.10123)
# Measure the influence valley on a RANDOMLY-INITIALIZED model at the same context
# length, then compare Step-0 vs pretrained vs standard-FT vs intervention.
# Expect: standard training DEEPENS the valley (their Fig. 3); the middle-weighted
# intervention FLATTENS it (lower peak/trough).
!python scripts/03_measure_influence.py --config $CFG --init-random
!python scripts/07_step0_compare.py     --config $CFG
import pandas as pd
display(pd.read_csv('outputs/tables/valley_depth_by_method.csv'))

In [ ]:
# 8. Show the figures inline
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('outputs/plots/*.png')):
    print(p); display(Image(p))

In [ ]:
# 7c. Weighting sweep (beta / sigma / edge-floor) -> find a NET middle-accuracy win
# Trains a fresh model per config (matched compute) and scores each on middle accuracy
# without regressing average/worst vs the 'standard' control.
!python scripts/08_sweep_intervention.py --config $CFG
import pandas as pd
display(pd.read_csv('outputs/tables/sweep_intervention.csv'))

In [ ]:
# 9. Claim 1 -- "Legible at birth": random-init fingerprint predicts trained accuracy
# Headline: Spearman over the architectural region (x<=0.7); recency shown to be learned.
!python scripts/09_fingerprint.py --config $CFG --label pythia-70m
import pandas as pd
display(pd.read_csv('outputs/tables/fingerprint_pythia-70m_summary.csv'))

# Depth law (causal): re-measure Step-0 + standard FT + fingerprint per model size.
for M, L in [("EleutherAI/pythia-70m","70m"), ("EleutherAI/pythia-160m","160m")]:  # add 410m if memory allows
    !python scripts/03_measure_influence.py --config $CFG --init-random --model $M
    !python scripts/04_train_standard.py    --config $CFG --model $M
    !python scripts/09_fingerprint.py       --config $CFG --label $L
!python scripts/09_fingerprint.py --config $CFG --depth-trend
display(pd.read_csv('outputs/tables/depth_trend.csv'))

In [ ]:
# 10. Claim 2 -- "Curable at birth": bake-off of init-time cures vs training-time fixes
# Searches a residual schedule that flattens the Step-0 valley (ramp_2to0.5 flattens;
# uniform scaling is inert), then compares standard / loss-weighting / resid_alpha / anchors.
!python scripts/10_cure_bakeoff.py --config $CFG
import pandas as pd
display(pd.read_csv('outputs/tables/cure_bakeoff.csv'))